# ZebraID — Training Notebook (Colab GPU)

**Run this on Google Colab with a T4/A100 GPU (Runtime → Change runtime type → GPU)**

This notebook produces the three core results for the paper:

| Config | Trained on | Evaluated on | Purpose |
|---|---|---|---|
| `baseline_a` | Pop A (GZGC) | Pop A | Within-population baseline |
| `baseline_x` | Pop A (GZGC) | Pop B (Grevy's) | Generalization gap |
| `zebraid` | Mixed A+B | Both | ⭐ Cross-population result |

Results are saved to `results/training_comparison.csv`.

---
**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells **1 → 13** in order.
3. All checkpoints are copied back to Google Drive in Cell 13.

In [ ]:
# ── 1. Mount Drive & Navigate to ZebraID ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Try common locations where the user might have placed ZebraID
possible_paths = [
    '/content/drive/MyDrive/ZebraID',
    '/content/drive/MyDrive/ZebraID-main',
    '/content/drive/My Drive/ZebraID',
    '/content/ZebraID',
]

target_dir = None
for p in possible_paths:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'pyproject.toml')):
        target_dir = p
        break

if target_dir:
    os.chdir(target_dir)
    print('✅ Working directory set to:', os.getcwd())
else:
    print('⚠️ Could not find ZebraID folder automatically.')
    print('Please run: os.chdir("/content/drive/MyDrive/YOUR_FOLDER_NAME")')
    print('Current dir:', os.getcwd())
    print('Drive contents:', os.listdir('/content/drive/MyDrive/'))

In [ ]:
# ── 2. Install dependencies & zebraid package ─────────────────────────────────
# Must run AFTER Cell 1 so we are inside the ZebraID directory
import os
print('Installing from:', os.getcwd())
assert os.path.exists('pyproject.toml'), 'ERROR: Not in ZebraID root! Re-run Cell 1 first.'

# Install all required packages (quiet mode, skip already-installed)
!pip install -q timm pytorch-metric-learning faiss-cpu httpx fastapi uvicorn scikit-learn pyyaml

# Install zebraid as an editable package from current directory
!pip install -e . --quiet

print('✅ Dependencies and zebraid package installed.')

In [ ]:
# ── 3. Align & link dataset paths ────────────────────────────────────────────
# Datasets are at ZebraID root. Loaders can find them automatically.
# This cell verifies they are present and creates data/ symlinks if needed.
import os, shutil

print('Current directory:', os.getcwd())
print('Contents:', [f for f in os.listdir('.') if not f.startswith('.')])

os.makedirs('data', exist_ok=True)

# GZGC dataset — check root or data/ subfolder
if os.path.exists('gzgc.coco'):
    print('✅ gzgc.coco found at root')
    if not os.path.exists('data/gzgc.coco'):
        try:
            os.symlink(os.path.abspath('gzgc.coco'), os.path.abspath('data/gzgc.coco'))
            print('   → symlinked to data/gzgc.coco')
        except Exception:
            pass  # loaders will use root path directly
elif os.path.exists('data/gzgc.coco'):
    print('✅ gzgc.coco found in data/')
else:
    print('❌ ERROR: gzgc.coco not found! Expected at ZebraID/gzgc.coco or ZebraID/data/gzgc.coco')

# Grevy's dataset — check root or data/ subfolder
if os.path.exists('labeled_mpala_grevys'):
    print('✅ labeled_mpala_grevys found at root')
    if not os.path.exists('data/labeled_mpala_grevys'):
        try:
            os.symlink(os.path.abspath('labeled_mpala_grevys'), os.path.abspath('data/labeled_mpala_grevys'))
            print('   → symlinked to data/labeled_mpala_grevys')
        except Exception:
            pass  # loaders will use root path directly
elif os.path.exists('data/labeled_mpala_grevys'):
    print('✅ labeled_mpala_grevys found in data/')
else:
    print('❌ ERROR: labeled_mpala_grevys not found! Expected at ZebraID/labeled_mpala_grevys')

In [ ]:
# ── 4. Validate datasets ──────────────────────────────────────────────────────
# Reload the module to pick up the correct CWD
import importlib
import zebraid.data.loaders as _ldrs
importlib.reload(_ldrs)
from zebraid.data.loaders import build_datasets

print('Validating dataset splits...')
for split in ('train', 'val', 'test'):
    ds_a, ds_b = build_datasets(split, transform=None)
    print(f'{split:5s} | GZGC: {ds_a.num_individuals:4d} indivs, {len(ds_a):5d} imgs  '
          f'| Grevy: {ds_b.num_individuals:3d} indivs, {len(ds_b):3d} imgs')
print('✅ Datasets validated.')

In [ ]:
# ── 5. Check GPU ──────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0))
    print('CUDA Version:', torch.version.cuda)
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print('⚠️ NO GPU DETECTED!')
    print('Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('Then re-run all cells from the top.')

In [ ]:
# ── 6. Config for this run ────────────────────────────────────────────────────
import os, yaml

# Reduce CUDA memory fragmentation (critical for large models like MegaDescriptor)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import importlib, zebraid.data.loaders as _ldrs
importlib.reload(_ldrs)

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Memory-safe settings for T4 (15 GB) + MegaDescriptor-L-384 (1.94 GB) ──────
# batch_size=16 at 384x384 causes OOM. Use micro-batch=4 + accum=4 instead.
cfg['training']['batch_size']  = 4       # micro-batch: fits in T4 VRAM
cfg['training']['accum_steps'] = 4       # gradient accumulate 4 steps -> effective batch 16
cfg['training']['use_wandb']   = False
cfg['training']['num_epochs']  = 30

with open('configs/default.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

eff = cfg['training']['batch_size'] * cfg['training']['accum_steps']
print('Config ready:')
print(f"  batch_size  = {cfg['training']['batch_size']}  (micro-batch per step)")
print(f"  accum_steps = {cfg['training']['accum_steps']}  -> effective batch = {eff}")
print(f"  num_epochs  = {cfg['training']['num_epochs']}")
print(f"  use_wandb   = {cfg['training']['use_wandb']}")
print(f"  checkpoint_dir = {cfg['paths']['checkpoint_dir']}")


In [ ]:
# ── 7. Train: Baseline A (GZGC only) ─────────────────────────────────────────
from zebraid.models.train import train

print('='*55)
print('Training Config: baseline_a  (Pop A → Pop A)')
print('='*55)

res_a = train(
    config_path='configs/default.yaml',
    mode='baseline_a',
    backbone_name='megadescriptor',
    split_seed=42,
)
ckpt_a = res_a['checkpoint_path']
print(f'\n✅ Saved: {ckpt_a}')
print(f'   Rank@1 Pop A = {res_a["rank1_a"]:.4f}  mAP Pop A = {res_a["map_a"]:.4f}')

In [ ]:
# ── 8. Evaluate Baseline A on Pop A val ──────────────────────────────────────
import torch
from zebraid.models.backbone import build_embedder
from zebraid.models.evaluate import compute_cmc_map
from zebraid.data.loaders import build_datasets
from zebraid.data.transforms import eval_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
t_eval = eval_transforms(384)
ds_a_val, ds_b_val = build_datasets('val', transform=t_eval)

model_a = build_embedder('megadescriptor', embedding_dim=512, pretrained=False, device=device)
model_a.load_state_dict(torch.load(ckpt_a, map_location=device, weights_only=True))
model_a.eval()

metrics_a = compute_cmc_map(model_a, ds_a_val, device, top_k=5)
print(f'Baseline A → Pop A val:')
print(f'  Rank@1 = {metrics_a["rank1"]:.4f}')
print(f'  Rank@5 = {metrics_a["rank5"]:.4f}')
print(f'  mAP    = {metrics_a["map"]:.4f}')

In [ ]:
# ── 9. Train: Baseline X (generalization gap) ─────────────────────────────────
print('='*55)
print('Training Config: baseline_x  (Pop A → Pop B)')
print('='*55)

res_x = train(
    config_path='configs/default.yaml',
    mode='baseline_x',
    backbone_name='megadescriptor',
    split_seed=42,
)
ckpt_x = res_x['checkpoint_path']
print(f'\n✅ Saved: {ckpt_x}')
print(f'   Rank@1 Pop A = {res_x["rank1_a"]:.4f}  Rank@1 Pop B = {res_x["rank1_b"]:.4f}')

In [ ]:
# ── 10. Train: ZebraID (mixed A+B) ───────────────────────────────────────────
print('='*55)
print('Training Config: zebraid  (Mixed A+B → Both)')
print('='*55)

res_zebraid = train(
    config_path='configs/default.yaml',
    mode='zebraid',
    backbone_name='megadescriptor',
    split_seed=42,
)
ckpt_zebraid = res_zebraid['checkpoint_path']
print(f'\n✅ Saved: {ckpt_zebraid}')
print(f'   Rank@1 Pop A = {res_zebraid["rank1_a"]:.4f}  Rank@1 Pop B = {res_zebraid["rank1_b"]:.4f}')

In [ ]:
# ── 11. Full comparison table ─────────────────────────────────────────────────
import csv, os, torch
from zebraid.models.backbone import build_embedder
from zebraid.models.evaluate import compute_cmc_map
from zebraid.data.loaders import build_datasets
from zebraid.data.transforms import eval_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
t_eval = eval_transforms(384)
ds_a_val, ds_b_val = build_datasets('val', transform=t_eval)

rows = []
configs = [
    ('baseline_a', ckpt_a),
    ('baseline_x', ckpt_x),
    ('zebraid',    ckpt_zebraid),
]

print(f'{'Config':12s} | Pop A R@1  R@5   mAP  | Pop B R@1  R@5   mAP')
print('-' * 60)

for config, ckpt in configs:
    model = build_embedder('megadescriptor', embedding_dim=512, pretrained=False, device=device)
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    model.eval()

    m_a = compute_cmc_map(model, ds_a_val, device, top_k=5)
    m_b = compute_cmc_map(model, ds_b_val, device, top_k=5)

    row = {
        'config':     config,
        'pop_a_r1':   round(m_a['rank1'], 4),
        'pop_a_r5':   round(m_a['rank5'], 4),
        'pop_a_map':  round(m_a['map'],   4),
        'pop_b_r1':   round(m_b['rank1'], 4),
        'pop_b_r5':   round(m_b['rank5'], 4),
        'pop_b_map':  round(m_b['map'],   4),
    }
    rows.append(row)
    print(f"{config:12s} | {m_a['rank1']:.4f}  {m_a['rank5']:.4f}  {m_a['map']:.4f} | {m_b['rank1']:.4f}  {m_b['rank5']:.4f}  {m_b['map']:.4f}")

os.makedirs('results', exist_ok=True)
csv_path = 'results/training_comparison.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
print(f'\n✅ Saved: {csv_path}')

In [ ]:
# ── 12. Also run ResNet50 ablation baseline ───────────────────────────────────
print('='*55)
print('Training Config: zebraid mode with ResNet50 backbone')
print('='*55)

res_resnet = train(
    config_path='configs/default.yaml',
    mode='zebraid',
    backbone_name='resnet50',
    split_seed=42,
)
ckpt_resnet_zebraid = res_resnet['checkpoint_path']
print(f'\n✅ ResNet50 ablation saved: {ckpt_resnet_zebraid}')
print(f'   Rank@1 Pop A = {res_resnet["rank1_a"]:.4f}  Rank@1 Pop B = {res_resnet["rank1_b"]:.4f}')

In [ ]:
# ── 13. Copy checkpoints & results to Google Drive ────────────────────────────
import shutil, os

# Save back to the same Drive folder so nothing is lost if Colab session ends
drive_root = os.getcwd()  # we are already inside /content/drive/MyDrive/ZebraID

print('Syncing outputs back to Google Drive...')
print('Drive root:', drive_root)

if os.path.exists('checkpoints'):
    print('✅ checkpoints/ already in Drive (we are running from Drive).')
if os.path.exists('results/training_comparison.csv'):
    print('✅ results/training_comparison.csv saved.')

# Also copy to a dedicated backup folder in Drive root
backup_dir = '/content/drive/MyDrive/ZebraID_results_backup'
os.makedirs(backup_dir, exist_ok=True)
if os.path.exists('results'):
    shutil.copytree('results', f'{backup_dir}/results', dirs_exist_ok=True)
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', f'{backup_dir}/checkpoints', dirs_exist_ok=True)

print(f'\n✅ Backup saved to: {backup_dir}')
print('Done! All training results are preserved in Google Drive.')